# new_pipeline_loo – smoke test
Checks the ElasticNet LOO pipeline on the first proteins of `data/protein_list.txt`.

Needs (inside `new_pipeline_loo/data/`): `transcriptomics_filtered.joblib`, `proteomics_matched.joblib`, `protein_list.txt`.
Writes to `data/results_smoke/` (never touches the real `results_<norm>/` folders).

In [ ]:
from pathlib import Path
import sys, subprocess, shlex
import pandas as pd
from joblib import load

ROOT = Path('.').resolve()
DATA_DIR, SRC_DIR = ROOT / 'data', ROOT / 'src'
RESULTS_DIR = DATA_DIR / 'results_smoke'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RNA_FILE = 'transcriptomics_filtered.joblib'          # same file the CLI uses by default
transcript_path = DATA_DIR / RNA_FILE
protein_path = DATA_DIR / 'proteomics_matched.joblib'
protein_list_path = DATA_DIR / 'protein_list.txt'
N_TEST = 2                                            # proteins to test
for p in (transcript_path, protein_path, protein_list_path):
    assert p.exists(), f'Missing {p}'
print('All inputs found in', DATA_DIR)

In [ ]:
X, Y = load(transcript_path), load(protein_path)
proteins = [p.strip() for p in protein_list_path.read_text().splitlines() if p.strip() and not p.startswith('#')]
test_proteins = proteins[:N_TEST]
print('X', X.shape, '| Y', Y.shape, '| testing', test_proteins)

## 1. In-process call (best-only, z-score)

In [ ]:
sys.path.insert(0, str(SRC_DIR))
from new_functions import run_protein_group
param_grid = {'alpha': [0.1, 0.5, 1.0], 'l1_ratio': [0.1, 0.5, 0.9]}
out_best = run_protein_group(proteins=test_proteins, pipeline='fixed_percentile', X=X, Y=Y,
                             param_grid=param_grid, percentile=5, n_splits=3, save_only_best=True,
                             output_dir=str(RESULTS_DIR / 'inprocess'), zscore=True, fit_intercept=False)
print(out_best[['protein', 'best_sample', 'best_abs_error', 'mse_all_others']])

## 2. CLI call, exactly as `new_run_enet.sh` runs it (`--full_output`)

In [ ]:
for norm in ['zscore', 'binning']:
    out_dir = RESULTS_DIR / f'results_{norm}'
    cmd = [sys.executable, str(SRC_DIR / 'new_pipeline.py'), '--start_idx', '0', '--group_size', str(N_TEST),
           '--pipeline', 'fixed_percentile', '--normalization', norm, '--percentile', '5', '--n_splits', '3',
           '--n_bins', '10', '--full_output', '--data_dir', str(DATA_DIR), '--rna_file', RNA_FILE,
           '--results_dir', str(out_dir), '--protein_list', str(protein_list_path)]
    print('CMD:', ' '.join(shlex.quote(c) for c in cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-600:]); assert r.returncode == 0, r.stderr[-2000:]

In [ ]:
for norm in ['zscore', 'binning']:
    out_dir = RESULTS_DIR / f'results_{norm}'
    for p in test_proteins:
        files = {k: sorted(out_dir.glob(f'{p}_{k}.*')) for k in ('best', 'LOO')}
        assert files['best'] and files['LOO'], f'missing outputs for {p} in {out_dir}'
    batch = sorted(out_dir.glob('batch_*'))
    print(norm, '->', [f.name for f in sorted(out_dir.iterdir())])
    loo = pd.read_parquet(sorted(out_dir.glob('*_LOO.parquet'))[0])
    print('LOO columns:', list(loo.columns), '| rows:', len(loo))